# Modal RGB petrography ML

Pipeline: pick images → draw labels → choose features → train a classifier → predict maps and area %.

Code lives in `pyPetrograph/` (purpose folders: `common/`, `image_processing/`, `labeling_ml/`; import stays `from pyPetrograph import …`).
Run cells **in order** from top to bottom.


## Setup — check packages

Assumes this notebook's kernel already uses your conda/venv (Cursor / VS Code). Checks packages needed by `pyPetrograph`; installs any missing ones into that env (often **PyQt5** for the label window).


In [ ]:
from pathlib import Path
import sys
ROOT = Path(".")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

from pyPetrograph import check_and_install_packages
_ = check_and_install_packages(install=True)


## Setup — imports

Loads the toolbox. Use **inline** plots for summary/preview in the notebook.
Labeling opens in a **separate desktop window** (so the Jupyter kernel does not crash).


In [ ]:
%matplotlib inline

from pathlib import Path

from pyPetrograph import (
    Session,
    select_images,
    launch_app,
    print_label_summary,
    print_train_metrics,
    run_feature_preview,
    run_train_cell,
    run_predict_cell,
)


## Settings

Edit values on `session` in the next cell. Defaults and tune tips:

**Classes**
- `session.class_names` — set here (e.g. 1 pores, 2 grains, 3 cements). Tune: edit the dict.

**Paths**
- `session.image_dir` — default `Path(".")` = folder that contains this notebook (when Jupyter was started there). Tune: `Path(r"/full/path/to/images")`.
- Accepted image types: `.tif` `.tiff` `.geotiff` `.jpg` `.jpeg` `.png` `.bmp` `.webp` `.gif`
- Outputs: `labels/` and `predictions/` stay beside each image folder; all models under notebook-level `models/` (universal + per-image). RGB is RAM-cached until kernel restart (no permanent `cache/` folder). Flat legacy companions are removed on select.

**Image load**
- `session.load_target_mp` — default `None` (full size). Example: `8.0` for huge scans.
- `session.max_side` — default `None`. Example: `4000`; if set, wins over `load_target_mp`.

**ML (active)**
- `session.method` — default `"lightgbm"`; options `"lightgbm"` | `"random_forest"` | `"xgboost"`. Tune: try `random_forest` if LightGBM is unavailable.
- `session.n_jobs` — default `4`. Tune: raise on multi-core machines (e.g. `8`).

**ML (advanced, commented in code)**
- `y_target` — default `128.0`. Tune: raise/lower if slides are very bright/dark.
- `median_size` — default `4`. Tune: `1` = no smooth after predict; larger = less speckly maps.

**Display (active)**
- `session.preview_dpi` — default `150`. Inline **feature preview** only. Tune: higher for sharper feature plots.
- `session.figure_dpi` — default `300`. Label summary + predict figures. Tune: lower for faster inline redraws.

**Labeling (package defaults)**
- `session.draw_mode` — default `"wand"` (or `"polygon"`).
- `session.wand_threshold` — default `70` (color similarity for magic wand; Label UI slider range **10–150**).

**Display / labeling (advanced, commented)**
- `preview_target_mp` — default `None` (screen-sized overview). Tune: e.g. `4.0` for a fixed size.
- `texture_window` — default `7`.
- `lbp_p` / `lbp_r` — defaults `8` / `1.0`.

**Train / predict** (set in those cells later)
- train `mode` — `"retrain"` | `"load"`
- train `scope` — `"universal"` | `"per_image"`
- predict `model_source` — `"universal"` | `"per_image"`


In [ ]:
# Parameter settings
session = Session()

# Predefined classes & class names
session.class_names = {
    1: "pores",
    2: "grains",
    3: "cements",
}
session.remember_settings_classes()
session.current_class = min(session.class_names)

# Image directory
session.image_dir = Path(".")   # default directory same as the notebook

# Image processing parameters
y_target = 128.0                # targeting brightness for all images
session.preview_dpi = 150
session.figure_dpi = 200 
session.draw_mode = "wand"      # Session default
session.wand_threshold = 70     # Session default

# Machine learning parameters
session.method = "lightgbm"     # "lightgbm" | "xgboost" | "random_forest"
session.n_jobs = 6              # CPU cores to use in ML training

# Print settings
print("Classes:", session.class_names)
print("IMAGE_DIR:", Path(session.image_dir).resolve())
print(
    "Load:",
    "native"
    if session.load_target_mp is None and session.max_side is None
    else f"target_mp={session.load_target_mp}, max_side={session.max_side}",
)
print(
    "method:", session.method,
    "| n_jobs:", session.n_jobs,
    "| preview_dpi:", session.preview_dpi,
    "| figure_dpi:", session.figure_dpi,
    "| draw:", session.draw_mode,
    "| wand:", session.wand_threshold,
)

## Select images

Opens a **file picker** via Qt in a **separate process** (macOS / Windows / Linux; same idea as the Label window — does not crash the Jupyter kernel).

1. Multi-select images in one folder.
2. Click **Add more** to pick from another subfolder (repeat as needed).
3. Click **Done** when the queue is complete.

After select: cleans flat legacy companions; RGB is RAM-cached until kernel restart (no permanent `cache/` folder). Prints use short paths relative to the notebook.

To add more later: `session = select_images(session, append=True)`.


In [ ]:
session = select_images(session)
# session = select_images(session, append=True)  # add from another folder

## Interactive UI: Class labeling for machine learning

Requires a queue from **Select images** above.

Opens a **Qt desktop popup** in a separate process (same backend idea as DMG-07’s `%matplotlib qt` pickers — **not** Tk):

- **Left:** Prev/Next, Save/Reload/Undo/Restart, Cycle class, Mode (default **wand**, threshold default **70**, slider **10–150**), editable class name + **Add label**, short status (current-class polygon count)
- **Right:** image + outline-only polygons; **minimap** in dedicated axes (lower-left)

**Enter** finishes a polygon · **Esc** starts a new one.

**Save** — only control that writes all on-screen labels to disk (overwrite). **Reload** — load saved labels from disk into memory (undoable). **Restart** — clear only the **current class** labels in the scene (polygons + pixels); does not reset class names or write disk. **Prev/Next** and closing the window do **not** autosave. **Undo** rewinds in-memory steps (including Restart and Reload). **Settings** cell is source of truth for knobs (`figure_dpi`, `draw_mode`, `wand_threshold`, etc.); `session.json` stores class names + `current_class` only. Then run Label summary here.

Needs **PyQt5** in conda env `work` (`conda install -c conda-forge pyqt` if missing).


In [ ]:
session = launch_app(session)

## Label summary — check what you labeled

Prints, for every selected image: polygon count, labeled pixel count, and classes present.

Shows **one inline figure per image**: gray background + colored polygons (`session.figure_dpi`, default 300). Also writes `labels/{stem}_labels.png` per image (figure export only).

On **Save**, labels folder gets `{stem}_polygons.geojson` + `{stem}_session.json` (class_names + current_class only; Settings knobs stay in the notebook). Train/reload rasterizes labels from polygons.

Run after labeling. Safe to re-run anytime.


In [ ]:
print_label_summary(session)

## Image-based features for ML classification

The next cell turns features **on/off** (`True`/`False`). Only `True` ones are used in training and prediction.

**Default if unset:** all **off** (`False`). Turn on what you need below.

| Toggle | Default | What it is | Why use it |
|--------|---------|------------|------------|
| `r`, `g`, `b` | off | Red / green / blue of each pixel | Basic color |
| `y` | off | Brightness (weighted gray from RGB) | Separates light vs dark areas |
| `local_std` | off | How much brightness varies in a small window | Rough vs smooth texture |
| `local_grad` | off | Edge strength (Sobel) | Boundaries / visual complexity |
| `lbp` | off | Local Binary Pattern code | Fine micro-texture pattern |
| `glcm` | off | Contrast + homogeneity **inside each drawn polygon** | Extra texture; slower |

**Preview colormaps:** `r`→Reds, `g`→Greens, `b`→Blues, `y`→gray; `local_std`/`local_grad`/`lbp`→viridis/plasma/inferno.

**Preview vs train:** the figure uses a screen-sized overview (or `preview_target_mp` if set in Settings). **Training uses the full working-resolution image** from load.

Preview plots **all queued images**, **one column** of maps (`figure_dpi`). Plots appear **inline** under the cell (not a Qt popup). Also saves `predictions/{stem}_features.png` per image.


In [ ]:
session.feature_toggles = {
    "r": True,
    "g": True,
    "b": True,
    # "y": False,          # may not be needed if RGB are used
    "local_std": True,
    "local_grad": True,
    "lbp": True,
    # "glcm": False,
}

run_feature_preview(session)

## Training ML models

Edit `mode` / `scope` in the next cell.

- **retrain** — trains from labeled pixels (features you toggled on). Writes under notebook-level `models/` (universal + per-image).
- **load** — file picker for a `.joblib` (does not retrain).
- **scope** — `"universal"` = one pooled model; `"per_image"` = one model per labeled image.
- Stratified **k-fold CV** (default 5 folds) runs during retrain; **metrics print in the Accuracy cell** below.
- Saved model is fit on **all** labeled pixels (not a holdout split).
- Folds run in parallel via `session.n_jobs`.

Images with no labels are skipped when retraining.


In [ ]:
train_results = run_train_cell(
    session,
    mode="retrain",     # "retrain" | "load"
    scope="universal",  # "universal" | "per_image"
    prompt_load=True,
)

## Model accuracy evaluation: Cross-validation

Prints stratified k-fold cross-validation (CV) metrics from the Train cell (`train_results`). Re-run after retraining.

In [ ]:
print_train_metrics(train_results, session=session)

## Predict classs for all selected images

Edit `model_source` in the next cell.

Runs on **all** queued images. Writes under `<image folder>/predictions/`:
- `*_pred.png`              # HSV RGB class map
- `*_pred_rgb.png`          # Original image + class-color overlay
- `*_fig_pred.png`          # Predict cell figure: class map + legend
- `*_fig_confidence.png`    # Predict cell figure: confidence map
- `*_confidence.npy`        # Array: confidence values for each pixel's prediction
- `*_fractions.csv`         # Table: predicted pixel fractions for each class per image

Shows **one inline figure per image** (`session.figure_dpi`; same content saved as `*_fig_*.png`). If a required model is missing, it **stops** and tells you what to do.


In [ ]:
predict_results = run_predict_cell(
    session,
    model_source="universal",  # "universal" | "per_image"
)